# Train an SO101-Nexus policy with PPO on MuJoCo Warp

PPO on the GPU-parallel MuJoCo Warp backend. Pick a task below and run every cell top to bottom.

## 1. Check the GPU

Runtime -> Change runtime type -> GPU.

In [ ]:
!nvidia-smi -L

## 2. Install

The example scripts live in `examples/`, not the wheel, so this clones the repo and installs the `warp` + `train` extras.

In [ ]:
!git clone --depth 1 https://github.com/johnsutor/so101-nexus.git
%cd so101-nexus
!pip uninstall -y so101-nexus 2>/dev/null || true
!pip install -q -e ".[warp,train]" "imageio[ffmpeg]"
# `pip install -e` registers the package via a .pth file that Python only
# reads when an interpreter starts. This kernel was already running before
# the install above, so `!python ...` cells (fresh subprocesses) see the
# package but in-kernel `import so101_nexus...` cells below will not unless
# `src/` is also put on sys.path explicitly, here.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

## 3. Choose the environment

PickLift needs the most steps to solve; the others solve in about a minute.

In [ ]:
# Options: WarpPickLift-v1, WarpTouch-v1, WarpLookAt-v1, WarpMove-v1
ENV_ID = "WarpPickLift-v1"
STEPS = {"WarpPickLift-v1": 30_000_000}.get(ENV_ID, 5_000_000)
print(f"Training {ENV_ID} for {STEPS:,} env steps")

## 4. Launch TensorBoard

Run before training so the dashboard picks up the new run.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## 5. Train

Trains the chosen environment end to end.

In [ ]:
!python examples/ppo_warp.py --exp-name colab --env-id {ENV_ID} \
    --total-timesteps {STEPS} --eval-freq 0

## 6. Evaluate the trained policy

Runs a deterministic evaluation across 512 parallel Warp environments.

In [ ]:
CKPT = f"runs/{ENV_ID}__colab__*/best_agent.pt"
!python examples/eval_warp.py --env-id {ENV_ID} --checkpoint "{CKPT}"

## 7. Watch a sample rollout

Renders one episode of the trained policy so you can see it work.


In [ ]:
# Render one deterministic rollout of the trained policy and play it inline.
import glob

from IPython.display import Video

from examples.ppo_warp import rollout_video_from_checkpoint

checkpoint = sorted(glob.glob(CKPT))[-1]
print(f"Rendering sample rollout from {checkpoint}")

metrics, video_path = rollout_video_from_checkpoint(
    checkpoint,
    ENV_ID,
    control_mode="pd_joint_delta_pos",
    episode_length=512,
    hidden_dim=256,
    seed=12345,
)
print(
    f"rollout success_rate={metrics['eval/success_rate']:.2f} "
    f"return={metrics['eval/return']:.2f} "
    f"ep_len={metrics['eval/ep_len']:.0f}"
)
Video(video_path)

## Next steps

- Try other seeds with `--seed <n>`; the default recipe was validated on seeds 1, 2, and 3.
- Full walkthrough: [Training with PPO](https://so101-nexus.com/docs/training/ppo).